# BlinkMimic T4：40 段动作逐个完整视频

指标和轨迹录制已经完成；本 Notebook 只下载已保存的 `eval_traj.npz` 并渲染完整 40 段动作，不重新运行策略评测。

每个动作单独生成一个 MP4，时长按轨迹中的 clip_len 计算；不拼接、不固定截取 3 秒。25 FPS 与 50 Hz 仿真轨迹精确对应。渲染 cell 实时显示动作进度，成功的视频会保留并可续跑；每个 run 渲染结束或失败后都会先上传已有视频和 manifest。

In [ ]:
from pathlib import Path
import os

RUNS = {
    'old_083958': {'run_id': '20261122_083958', 'gpu': '0'},
    'new_105913': {'run_id': '20261122_105913', 'gpu': '1'},
}
REPO_DIR = Path('/kaggle/working/proj2-humanoid-mimic/third_party/blinkmimic')
REPO_URL = 'https://github.com/fuyichen0725/blinkmimic.git'
COMMIT = 'd264658d6c9ce61fb52f9f0f8985a369f3ccc409'
BASE_RESULTS = Path('/kaggle/working/proj2-humanoid-mimic/results/blinkmimic_t4_ddp')
HF_REPO_NAME = 'proj2-blinkmimic-t4'
HF_REPO = 'A2bao/proj2-blinkmimic-t4'
VIDEO_FPS = 25
VIDEO_WIDTH = 640
VIDEO_HEIGHT = 360
PROJECT_DIR = Path('/kaggle/working/proj2-humanoid-mimic')
PROJECT_GITHUB = 'https://github.com/magickkdd/g1-mimic-t4.git'
RENDER_SCRIPT = PROJECT_DIR / 'scripts' / 'render_each_motion.py'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or HF_TOKEN
except Exception as exc:
    print(f'kaggle_secrets unavailable: {exc}')
for item in RUNS.values():
    item['root'] = BASE_RESULTS / f"run_{item['run_id']}"
print('HF_TOKEN=' + ('set' if HF_TOKEN else 'MISSING'))
print({name: item['run_id'] for name, item in RUNS.items()})


In [ ]:
import subprocess
def run(command, cwd=None, check=True):
    print('$', ' '.join(str(x) for x in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if not (PROJECT_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', PROJECT_GITHUB, str(PROJECT_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
if head != COMMIT:
    run(['git', 'reset', '--hard', COMMIT], cwd=REPO_DIR)
run(['uv', 'sync', '--frozen'], cwd=REPO_DIR)
run(['bash', 'tools/download_data.sh'], cwd=REPO_DIR)
print('repo_and_motion_data=READY')


In [ ]:
import shutil
import subprocess
import sys
try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import HfApi, hf_hub_download
if not HF_TOKEN:
    raise RuntimeError('缺少 HF_TOKEN')
api = HfApi(token=HF_TOKEN)
assert api.whoami()['name'] == 'A2bao', 'Unexpected Hugging Face account'
for name, item in RUNS.items():
    root = item['root']
    root.mkdir(parents=True, exist_ok=True)
    target = root / 'eval_traj.npz'
    if not target.exists():
        remote = f"runs/run_{item['run_id']}/eval/eval_traj.npz"
        path = hf_hub_download(repo_id=HF_REPO, filename=remote, local_dir=str(root / 'hf_download'), token=HF_TOKEN)
        shutil.copy(path, target)
    print(f"{name}: traj={target} size={target.stat().st_size / 1e6:.1f} MB")


In [ ]:
import numpy as np
for name, item in RUNS.items():
    with np.load(item['root'] / 'eval_traj.npz', allow_pickle=False) as traj:
        names = [str(x) for x in traj['names']]
        lengths = traj['clip_len'].astype(int)
    assert len(names) == 40 and len(lengths) == 40, f'{name}: expected 40 motions'
    assert len(set(names)) == 40, f'{name}: duplicate motion names'
    print(f'{name}: 40 clips, duration range {min((n - 1) / 50 for n in lengths):.2f}-{max((n - 1) / 50 for n in lengths):.2f}s')
    item['clips'] = names


In [ ]:
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
def render_one(name, item):
    root = item['root']
    output_dir = root / 'videos'
    output_dir.mkdir(parents=True, exist_ok=True)
    renderer = RENDER_SCRIPT
    render_py = REPO_DIR / 'tools' / 'render.py'
    command = [sys.executable, str(renderer), str(root / 'eval_traj.npz'), str(render_py), str(output_dir), '--fps', str(VIDEO_FPS), '--width', str(VIDEO_WIDTH), '--height', str(VIDEO_HEIGHT), '--title', f"G1-Mimic-T4 {item['run_id']}", '--clean']
    env = os.environ.copy()
    env['MUJOCO_GL'] = 'egl'
    env['CUDA_VISIBLE_DEVICES'] = item['gpu']
    start = time.perf_counter()
    print(f'{name}: starting 40 per-motion renders on GPU {item["gpu"]}', flush=True)
    log_path = root / 'eval_render_each_motion_stdout.txt'
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(f'[{name}] {line}', end='', flush=True)
            log.write(line)
        result_code = process.wait()
    result = subprocess.CompletedProcess(command, result_code)
    return name, result, time.perf_counter() - start, output_dir
with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(lambda pair: render_one(*pair), RUNS.items()))
render_failures = []
for name, result, seconds, output_dir in results:
    videos = list(output_dir.glob('*.mp4'))
    print(f'{name}: exit={result.returncode} seconds={seconds:.1f} individual_videos={len(videos)}')
    if result.returncode != 0:
        print(result.stderr[-5000:])
        render_failures.append(name)
    elif len(videos) != 40 or not (output_dir / 'manifest.json').exists():
        render_failures.append(name)
        print(f'{name}: incomplete outputs, preserving {len(videos)} completed MP4s')
    else:
        print(f'{name}: all 40 motion videos and duration manifest verified')


In [ ]:
from IPython.display import Video, display
for name, item in RUNS.items():
    root = item['root']
    output_dir = root / 'videos'
    api.upload_folder(folder_path=str(output_dir), path_in_repo=f"runs/run_{item['run_id']}/eval/videos", repo_id=HF_REPO, repo_type='model', commit_message=f"Upload per-motion render outputs for {item['run_id']}")
    log = root / 'eval_render_each_motion_stdout.txt'
    api.upload_file(path_or_fileobj=str(log), path_in_repo=f"runs/run_{item['run_id']}/eval/{log.name}", repo_id=HF_REPO, repo_type='model')
    print(f"{name}: uploaded_mp4_count={len(list(output_dir.glob('*.mp4')))}")
    if len(list(output_dir.glob('*.mp4'))) == 40 and (output_dir / 'manifest.json').exists():
        legacy = f"runs/run_{item['run_id']}/eval/eval_video_all40.mp4"
        try:
            api.delete_file(path_in_repo=legacy, repo_id=HF_REPO, repo_type='model', commit_message=f"Remove truncated aggregate for {item['run_id']}")
            print(f'[hf] removed obsolete {legacy}')
        except Exception as exc:
            if '404' not in str(exc) and 'Entry not found' not in str(exc):
                raise
if render_failures:
    raise RuntimeError(f'Render incomplete for: {render_failures}; completed videos and manifests were uploaded and can be reused on rerun')
print('40_separate_complete_motion_videos_upload=PASS')
